In [ ]:
import numpy as np
import tensorflow as tf

In [ ]:
X = np.memmap("X_AllStepsR_f32_norm.memmap", dtype=np.float32, mode="r", shape=(183199, 101, 75, 40))
# X = np.memmap("X_AllStepsR_f32_normMM.memmap", dtype=np.float32, mode="r", shape=(183199, 101, 75, 40))
Y = np.load("Y_AllStepsR.npy")       # class 0 -> 150
XRef = np.load("XRef_AllStepsR_f32_norm.npy")
YRef = np.load('YRef_AllStepsR.npy') # class 233 -> 247
YRef -= 80                           # class 153 -> 167

In [ ]:
X.shape, XRef.shape

### V6. 1D Inception with gradient reversal

In [ ]:
BATCH_SIZE = 32
VAL_SPLIT = 0.05

In [ ]:
N = len(Y)

def batch_generator(X, Y, ids, batch_size=BATCH_SIZE, shuffle=True, flat=True):
    N = len(ids)
    while True:
        if shuffle:
            np.random.shuffle(ids)
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = ids[start:end]
            X_ = X[batch]
            Y_ = Y[batch, 0:1]
            Y_1 = Y[batch, 1:2]
            Y_2 = Y[batch, 2:3]

            if flat:
                X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            
            yield X_, {"dense": Y_, "dense_1": Y_1, "dense_2": Y_2}

indices = np.arange(N)
np.random.shuffle(indices)
id_train = indices[:int(N*(1-VAL_SPLIT))]
id_valid = indices[int(N*(1-VAL_SPLIT)):]

gen_train = batch_generator(X, Y, id_train, batch_size=BATCH_SIZE, shuffle=True)
gen_valid = batch_generator(X, Y, id_valid, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
A, B = next(gen_train)

In [ ]:
import inceptiongradreverse
model = inceptiongradreverse.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),

    loss={
        "dense": tf.keras.losses.SparseCategoricalCrossentropy(),
        "dense_1": tf.keras.losses.SparseCategoricalCrossentropy(),
        "dense_2": tf.keras.losses.SparseCategoricalCrossentropy()
    },

    loss_weights={
        "dense": 1.0,
        "dense_1": 1.0,
        "dense_2": 1.0
    },

    metrics={
        "dense": tf.keras.metrics.SparseCategoricalAccuracy(name="acc")
    }
)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(id_train)//BATCH_SIZE//2, # Smaller "fake epochs" for faster feedback
    validation_data=gen_valid,
    validation_steps=len(id_valid)//BATCH_SIZE,
    batch_size=BATCH_SIZE,
    epochs=100,
    callbacks=[checkpoint]
)

In [ ]:
model.save('last_model.keras')
# model.save_weights('best_model.weights.h5')
# model.load_weights('best_model.weights.h5')

In [ ]:
model.load_weights('best_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

In [ ]:
model.load_weights('last_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

### V5. V1 architecture but contrastive loss

In [ ]:
BATCH_SIZE = 128 # larger batch size for contrastive loss
# BATCH_SIZE = 32
VAL_SPLIT = 0.05

In [ ]:
N = len(Y)

def batch_generator(X, Y, ids, batch_size=BATCH_SIZE, shuffle=True, flat=True):
    N = len(ids)
    while True:
        if shuffle:
            np.random.shuffle(ids)
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = ids[start:end]
            X_ = X[batch]
            Y_ = Y[batch, 0:1]

            if flat:
                X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            
            # yield X_, Y_
            yield X_, {"dense": Y_, "lambda": Y_}

indices = np.arange(N)
np.random.shuffle(indices)
id_train = indices[:int(N*(1-VAL_SPLIT))]
id_valid = indices[int(N*(1-VAL_SPLIT)):]

gen_train = batch_generator(X, Y, id_train, batch_size=BATCH_SIZE, shuffle=True)
gen_valid = batch_generator(X, Y, id_valid, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
A, B = next(gen_train)

In [ ]:
import inceptionembed
model = inceptionembed.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])

In [ ]:
class SupervisedContrastiveLoss(tf.keras.losses.Loss):
    def __init__(self, temperature=0.07, name="supcon"):
        super().__init__(name=name)
        self.temperature = temperature

    def call(self, labels, features):
        labels = tf.reshape(labels, [-1])
        labels = tf.cast(labels, tf.int32)

        features = tf.math.l2_normalize(features, axis=1)

        logits = tf.matmul(features, features, transpose_b=True)
        logits = logits / self.temperature

        mask = tf.equal(
            tf.expand_dims(labels, 1),
            tf.expand_dims(labels, 0)
        )

        logits_mask = tf.ones_like(mask, dtype=tf.float32) - tf.eye(tf.shape(labels)[0])
        mask = tf.cast(mask, tf.float32) * logits_mask

        exp_logits = tf.exp(logits) * logits_mask
        log_prob = logits - tf.math.log(tf.reduce_sum(exp_logits, axis=1, keepdims=True) + 1e-9)

        mean_log_prob_pos = tf.reduce_sum(mask * log_prob, axis=1) / (
            tf.reduce_sum(mask, axis=1) + 1e-9
        )

        loss = -tf.reduce_mean(mean_log_prob_pos)
        return loss

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),

    loss={
        "dense": tf.keras.losses.SparseCategoricalCrossentropy(),
        "lambda": SupervisedContrastiveLoss()
    },

    loss_weights={
        "dense": 1.0,
        "lambda": 0.2
    },

    metrics={
        "dense": tf.keras.metrics.SparseCategoricalAccuracy(name="acc")
    }
)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(id_train)//BATCH_SIZE//2, # Smaller "fake epochs" for faster feedback
    validation_data=gen_valid,
    validation_steps=len(id_valid)//BATCH_SIZE,
    batch_size=BATCH_SIZE,
    epochs=100,
    callbacks=[checkpoint]
)

In [ ]:
model.save('last_model.keras')
# model.save_weights('best_model.weights.h5')
# model.load_weights('best_model.weights.h5')

In [ ]:
model.load_weights('best_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

In [ ]:
model.load_weights('last_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

### V4. V1 architecture but triplet loss addition (not real triplet system, next version for this...)

In [ ]:
BATCH_SIZE = 32
VAL_SPLIT = 0.05

In [ ]:
N = len(Y)

def batch_generator(X, Y, ids, batch_size=BATCH_SIZE, shuffle=True, flat=True):
    N = len(ids)
    while True:
        if shuffle:
            np.random.shuffle(ids)
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = ids[start:end]
            X_ = X[batch]
            Y_ = Y[batch, 0:1]

            if flat:
                X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            
            # yield X_, Y_
            yield X_, {"dense": Y_, "lambda": Y_}

indices = np.arange(N)
np.random.shuffle(indices)
id_train = indices[:int(N*(1-VAL_SPLIT))]
id_valid = indices[int(N*(1-VAL_SPLIT)):]

gen_train = batch_generator(X, Y, id_train, batch_size=BATCH_SIZE, shuffle=True)
gen_valid = batch_generator(X, Y, id_valid, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
A, B = next(gen_train)

In [ ]:
import inceptionembed
model = inceptionembed.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])

In [ ]:
def triplet_loss(a,p,n,margin=0.3):
    dap = tf.reduce_sum(tf.square(a-p), axis=1)
    dan = tf.reduce_sum(tf.square(a-n), axis=1)
    return tf.reduce_mean(tf.maximum(dap - dan + margin, 0.0))

In [ ]:
def batch_triplet_loss(y_true, y_pred, margin=0.3):
    labels = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    emb = tf.math.l2_normalize(y_pred, axis=1)

    # pairwise distances
    dot = tf.matmul(emb, emb, transpose_b=True)
    sq = tf.linalg.diag_part(dot)
    dist = tf.expand_dims(sq,1) - 2*dot + tf.expand_dims(sq,0)
    dist = tf.maximum(dist, 0.0)

    same = tf.equal(tf.expand_dims(labels,1), tf.expand_dims(labels,0))
    diff = tf.logical_not(same)

    same = tf.cast(same, tf.float32)
    diff = tf.cast(diff, tf.float32)

    # hardest positive
    pos = tf.reduce_max(dist * same, axis=1)

    # hardest negative
    maxdist = tf.reduce_max(dist, axis=1, keepdims=True)
    neg = dist + maxdist * (1.0 - diff)
    neg = tf.reduce_min(neg, axis=1)

    loss = tf.maximum(pos - neg + margin, 0.0)
    return tf.reduce_mean(loss)

In [ ]:
'''
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="acc")
    ]
)
'''
model.compile(
    optimizer=tf.keras.optimizers.Adam(),

    loss={
        "dense": tf.keras.losses.SparseCategoricalCrossentropy(),
        "lambda": batch_triplet_loss
    },

    loss_weights={
        "dense": 1.0,
        "lambda": 0.2
    },

    metrics={
        "dense": tf.keras.metrics.SparseCategoricalAccuracy(name="acc")
    }
)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(id_train)//BATCH_SIZE//2, # Smaller "fake epochs" for faster feedback
    validation_data=gen_valid,
    validation_steps=len(id_valid)//BATCH_SIZE,
    batch_size=BATCH_SIZE,
    epochs=100,
    callbacks=[checkpoint]
)

In [ ]:
model.save('last_model.keras')
# model.save_weights('best_model.weights.h5')
# model.load_weights('best_model.weights.h5')

In [ ]:
model.load_weights('best_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)

In [ ]:
model.load_weights('last_model.keras')
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)